<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Introduction" data-toc-modified-id="Introduction-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Introduction</a></span></li><li><span><a href="#Prepare-the-data" data-toc-modified-id="Prepare-the-data-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Prepare the data</a></span><ul class="toc-item"><li><span><a href="#Load-packages" data-toc-modified-id="Load-packages-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Load packages</a></span></li><li><span><a href="#explore-the-data-directory" data-toc-modified-id="explore-the-data-directory-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>explore the data directory</a></span></li><li><span><a href="#Load-the-rating-data-to-pandas-dataframe" data-toc-modified-id="Load-the-rating-data-to-pandas-dataframe-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Load the rating data to pandas dataframe</a></span></li><li><span><a href="#Load-the-item-data" data-toc-modified-id="Load-the-item-data-2.4"><span class="toc-item-num">2.4&nbsp;&nbsp;</span>Load the item data</a></span></li><li><span><a href="#Convert-data-into-user-item-rating-matrix-and-user-item-binary-rating-matrix" data-toc-modified-id="Convert-data-into-user-item-rating-matrix-and-user-item-binary-rating-matrix-2.5"><span class="toc-item-num">2.5&nbsp;&nbsp;</span>Convert data into user-item rating matrix and user-item binary rating matrix</a></span><ul class="toc-item"><li><span><a href="#check-the-number-of-users-and-items" data-toc-modified-id="check-the-number-of-users-and-items-2.5.1"><span class="toc-item-num">2.5.1&nbsp;&nbsp;</span>check the number of users and items</a></span></li><li><span><a href="#Construct-the-user-time-rating-matrix" data-toc-modified-id="Construct-the-user-time-rating-matrix-2.5.2"><span class="toc-item-num">2.5.2&nbsp;&nbsp;</span>Construct the user-time rating matrix</a></span></li></ul></li></ul></li><li><span><a href="#Filtering" data-toc-modified-id="Filtering-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Filtering</a></span><ul class="toc-item"><li><span><a href="#Load-the-cosine_similarity-package-to-calculate-the-pairwise-cosine-similarity-between-users" data-toc-modified-id="Load-the-cosine_similarity-package-to-calculate-the-pairwise-cosine-similarity-between-users-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Load the cosine_similarity package to calculate the pairwise cosine similarity between users</a></span></li><li><span><a href="#calculate-user-similarity" data-toc-modified-id="calculate-user-similarity-3.2"><span class="toc-item-num">3.2&nbsp;&nbsp;</span>calculate user similarity</a></span></li><li><span><a href="#Create-a-dataframe-to-store-the-prediction-of-user-ratings-on-un-rated-items" data-toc-modified-id="Create-a-dataframe-to-store-the-prediction-of-user-ratings-on-un-rated-items-3.3"><span class="toc-item-num">3.3&nbsp;&nbsp;</span>Create a dataframe to store the prediction of user ratings on un-rated items</a></span></li><li><span><a href="#calculate-the-value-in-data_pre" data-toc-modified-id="calculate-the-value-in-data_pre-3.4"><span class="toc-item-num">3.4&nbsp;&nbsp;</span>calculate the value in data_pre</a></span></li></ul></li></ul></div>

# Centered Cosine Similarity-Based Collaborative Filtering

## Introduction
Learn Centered Cosine Similarity-Based Collaborative Filtering

## Prepare the data

### Load packages

In [1]:
import numpy as np
import pandas as pd

### explore the data directory

Move into the data directory

In [ ]:
cd ml-100k/

List the files in the ml-100k folder

In [ ]:
ls

Print the first 10 lines in u.data:

In [ ]:
!head u.data

### Load the rating data to pandas dataframe

In [ ]:
names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('u.data', sep='\t', names=names)
df.head()

### Load the item data

In [ ]:
i_cols = ['movie id', 'movie title' ,'release date','video release date', 'IMDb URL', 'unknown', 'Action', 'Adventure',
 'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
items = pd.read_csv('u.item', sep='|', names=i_cols, encoding='latin-1')

items.head()

### Convert data into user-item rating matrix and user-item binary rating matrix

#### check the number of users and items

In [ ]:
n_users = df.user_id.unique().shape[0]
n_items = df.item_id.unique().shape[0]
print(f"{n_users} users")
print(f"{n_items} items")

#### Construct the user-time rating matrix

In [ ]:
ratings = np.zeros((n_users, n_items))
ratingsBinary = np.zeros((n_users, n_items))

In [ ]:
for row in df.itertuples():
    ratings[row[1]-1, row[2]-1] = row[3]
    ratingsBinary[row[1]-1, row[2]-1] = 1
print(ratings)
print(ratingsBinary)

Convert the 'ratings' to a dataframe

In [ ]:
ratings_df = pd.DataFrame(ratings)

In [ ]:
ratings_df.head()

calculate the average rating for each user

In [ ]:
ratings_userRateNum = ratingsBinary.sum(axis = 1)
ratings_sum = ratings.sum(axis = 1)

In [ ]:
ratings_userAvg = ratings_sum/ratings_userRateNum

subtract the average rating for each rating.

In [ ]:
ratings_subtractMean = np.zeros((n_users, n_items))

In [ ]:
for row in df.itertuples():
    ratings_subtractMean[row[1]-1, 
                         row[2]-1] = row[3] - ratings_userAvg[row[1]-1]

In [ ]:
ratings_subtractMean_df = pd.DataFrame(ratings_subtractMean)

In [ ]:
ratings_subtractMean_df.head()

## Filtering

### Load the cosine_similarity package to calculate the pairwise cosine similarity between users

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

### calculate user similarity

In [ ]:
userSimilarity = cosine_similarity(ratings_subtractMean,
                                   ratings_subtractMean)

convert the numpy array to dataframe

In [ ]:
data_ubs_fast = pd.DataFrame(userSimilarity)

In [ ]:
data_ubs_fast.head()

In [ ]:
data_ubs_fast

### Create a dataframe to store the prediction of user ratings on un-rated items

In [ ]:
data_pre = pd.DataFrame(index=range(0, n_users), 
                        columns=items['movie title'])

### calculate the value in data_pre
> it will take very long time to run, for my mac pro, it takes 1 hour

In [ ]:
# small value, avoid divided by 0
epsilon = 1e-9

# loop through each user i and each items j
# only try the first three user here to prevent to run for a long time
for i in range(0,3):
    for j in range(0, len(data_pre.columns)):
        if ratings_df.iloc[i,j] > 0:
            data_pre.iloc[i,j]=0
        else:
            # select users who rated j with a mask
            mask_rated_users = ratings_df[j]>0
            # select k nearest neighbour's IDs and similarities
            top_neighbours = data_ubs_fast[mask_rated_users]\
                                .iloc[:, i]\
                                .sort_values(ascending=False)[1:10]\
                                .index
            top_neighbours_sim = data_ubs_fast[mask_rated_users]\
                                .iloc[:, i]\
                                .sort_values(ascending=False)[1:10]\
                                .values
            # select K nearest neighbour's ratings on j
            neighbours_ratings = ratings_df.iloc[top_neighbours, j]
            # final prediction on j for user i by using weighted sum formula
            data_pre.iloc[i,j] = sum(neighbours_ratings*top_neighbours_sim)/(sum(top_neighbours_sim)+epsilon)
                                

In [ ]:
data_pre.head(3)

In [ ]:
# Create a dataframe to store the final top-6 recommendations
data_recommend = pd.DataFrame(index=data_pre.index, 
                              columns = ['1', '2', '3', '4', '5', '6'])

# Make recommendations to each user by using 'for' 
# loop statement to sort the predictions for each user
for i in range(0, len(data_pre.index)):
    data_recommend.iloc[i, 0:] = data_pre.iloc[i,:]\
                                .sort_values(ascending=False)[0:6]\
                                .index\
                                .transpose()
data_recommend.head()